# ASR Baseline (Whisper) – Zeroth Korean

Zeroth Korean **test set**을 기반으로 Whisper ASR 베이스라인을 빠르게 확인하기 위한 노트북입니다.

- 입력 오디오: `~/data_zeroth/audio/**/**/*.wav` (하위 디렉터리를 포함한 모든 WAV 파일)
- 정답 스크립트: `~/data_zeroth/refs.csv` (파일명 기준으로 오디오와 매칭)

---

## 목적
1) **Whisper 모델별 성능(WER/CER) 및 실행시간 저장**(경진대회/수업 실험 기록)

---

## 라이선스/고지 (Zeroth Korean)
- 본 노트북은 **Zeroth Korean 데이터셋의 test set**을 실험용으로 사용합니다.
- 데이터셋의 원본 라이선스 및 이용조건은 데이터 제공처/원본 문서를 따릅니다.
- 본 실험 결과(전사/점수)는 교육 목적의 벤치마크 기록이며, 데이터 재배포는 하지 않습니다.

In [1]:
!pip install -U "transformers>=4.41" accelerate jiwer soundfile librosa pandas

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os, time, json
from pathlib import Path

import pandas as pd
import torch
from accelerate import Accelerator
from transformers import pipeline as hf_pipeline, AutoProcessor, AutoModelForSpeechSeq2Seq
from jiwer import wer, cer


In [3]:
from pathlib import Path
import os

HOME = Path.home()
BASE_DIR = Path.cwd()
BASE = HOME / "data_zeroth"
DATA_AUDIO_DIR = HOME / "data_zeroth" / "test_data_01"
REF_CSV = BASE / "refs.csv"

# 여러 모델을 여기서 바꿔가며 비교
MODEL_IDS = [
    "openai/whisper-tiny",
    "openai/whisper-base",
    "openai/whisper-small",
    # "openai/whisper-medium",  # 느리면 주석
]

LANGUAGE = "korean"
TASK = "transcribe"

MAX_FILES = int(os.environ.get("MAX_FILES", "50"))
SEED = int(os.environ.get("SEED", "42"))

OUT_DIR = Path(os.environ.get("OUT_DIR", BASE_DIR / "outputs_asr"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_CSV = OUT_DIR / "benchmark_results.csv"

print("DATA_AUDIO_DIR:", DATA_AUDIO_DIR)
print("REF_CSV:", REF_CSV)
print("MODELS:", MODEL_IDS)
print("MAX_FILES:", MAX_FILES, "| SEED:", SEED)
print("RESULT_CSV:", RESULT_CSV)

assert DATA_AUDIO_DIR.exists(), f"Missing audio dir: {DATA_AUDIO_DIR}"
assert REF_CSV.exists(), f"Missing refs.csv: {REF_CSV}"


DATA_AUDIO_DIR: /home/data/bootcamp/user2/data_zeroth/test_data_01
REF_CSV: /home/data/bootcamp/user2/data_zeroth/refs.csv
MODELS: ['openai/whisper-tiny', 'openai/whisper-base', 'openai/whisper-small']
MAX_FILES: 50 | SEED: 42
RESULT_CSV: /home/data/bootcamp/user2/gits/espnet-notebook-202601.new/bootcamp/outputs_asr/benchmark_results.csv


In [4]:
df_ref = pd.read_csv(REF_CSV)

# refs.csv 컬럼: file/ref로 통일
if "ref" not in df_ref.columns and "transcript" in df_ref.columns:
    df_ref = df_ref.rename(columns={"transcript": "ref"})

assert "file" in df_ref.columns, f"refs.csv needs 'file' column. got={df_ref.columns.tolist()}"
assert "ref" in df_ref.columns, f"refs.csv needs 'ref' (or transcript) column. got={df_ref.columns.tolist()}"

# file은 basename으로 맞추기
df_ref["file"] = df_ref["file"].astype(str).apply(lambda x: os.path.basename(x))
df_ref["ref"] = df_ref["ref"].astype(str)

df_ref.head()


,file,ref
0,104_003_0019.wav,지난해 삼 월 김 전 장관의 동료인 장동련 홍익대 교수가 민간 자문단장으로 위촉되면...
1,104_003_0193.wav,그 바람에 나의 몸도 겹쳐 쓰러지며 한창 피어난 노란 동백꽃 속으로 폭 파묻혀 버렸다
2,104_003_0253.wav,현재 백화점과 영화관 등은 오픈해 영업하고 있고 테마파크 및 아파트 등의 공사는 이...
3,104_003_0280.wav,또한 박 전 대통령 측은 앞으로 검찰 수사와 관련해 수사 내용에 대한 입장뿐 아니라...
4,104_003_0294.wav,한 잔의 커피가 소비자에게 전해지기 위해선 여러 차례의 공정을 거쳐야 한다


In [5]:
import random, glob

audio_files_all = sorted(Path(p) for p in glob.glob(os.path.join(DATA_AUDIO_DIR, "**", "*.wav"), recursive=True))
assert len(audio_files_all) > 0, f"No wav files in {DATA_AUDIO_DIR}"

# 2) basename -> fullpath 매핑 (중복 basename이 있으면 마지막이 덮어씀; Zeroth는 보통 유니크)
wav_map = {p.name: str(p) for p in audio_files_all}

# refs.csv에 있는 파일만 대상으로 (정답 없는 파일 제외)
ref_set = set(df_ref["file"].tolist())
audio_files = [p for p in audio_files_all if p.name in ref_set]

assert len(audio_files) > 0, "No overlap between refs.csv and audio wav files."

# 빠른 확인용 샘플링 (재현 가능). 직접 실험해 볼때는 사용하지 않는 코드입니다.
#rng = random.Random(SEED)
#if MAX_FILES > 0 and len(audio_files) > MAX_FILES:
#    audio_files = rng.sample(audio_files, MAX_FILES)

audio_files = sorted(audio_files)  # 출력 보기 좋게
print("#audio used:", len(audio_files), "/", len(audio_files_all))
print("sample:", audio_files[0])


#audio used: 456 / 456
sample: /home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_0019.wav


In [6]:
df_eval_by_model = {}  # model_id -> df_eval (columns: file, ref, hyp, model_id, wav_path)
df_hyp_by_model  = {}  # model_id -> df_hyp (columns: file, hyp, model_id)


In [7]:
def transcribe_one_model(model_id: str, files, df_ref: pd.DataFrame):
    import os, time
    from pathlib import Path
    import pandas as pd
    import torch
    from transformers import pipeline as hf_pipeline, AutoProcessor, AutoModelForSpeechSeq2Seq
    from jiwer import wer, cer

    BATCH_SIZE = 8

    # device/dtype (함수 내부에서 확정)
    if torch.cuda.is_available():
        device = 0
        visible = os.environ.get("CUDA_VISIBLE_DEVICES", "(not set)")
        dtype = torch.float16
    else:
        device = -1
        visible = "(cpu)"
        dtype = torch.float32

    print(f"MODEL: {model_id} | CUDA_VISIBLE_DEVICES: {visible} | pipeline device: {device} | dtype: {dtype}")

    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, dtype=dtype)
    if torch.cuda.is_available():
        model = model.to("cuda")

    asr = hf_pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        device=device,
    )

    # files: Path 리스트든 str 리스트든 모두 처리
    file_strs = [str(p) for p in files]

    t0 = time.time()
    outs = asr(
        file_strs,
        batch_size=BATCH_SIZE,
        generate_kwargs={"language": LANGUAGE, "task": TASK, "temperature": 0.0},
    )
    elapsed = round(time.time() - t0, 2)

    df_hyp = pd.DataFrame(
        [{"file": os.path.basename(p), "hyp": o["text"].strip()} for p, o in zip(file_strs, outs)]
    )
    df_hyp["model_id"] = model_id

    # merge + 매칭 체크
    df_eval = df_ref.merge(df_hyp, on="file", how="inner")
    df_eval["model_id"] = model_id

    print("ref rows:", len(df_ref), "| hyp rows:", len(df_hyp), "| matched:", len(df_eval))
    if len(df_eval) == 0:
        raise ValueError("No matched rows. Check df_ref['file'] == basename(wav filename).")

    w = wer(df_eval["ref"].tolist(), df_eval["hyp"].tolist())
    c = cer(df_eval["ref"].tolist(), df_eval["hyp"].tolist())

    return df_hyp, df_eval, elapsed, w, c


In [8]:
all_results = []

for model_id in MODEL_IDS:
    print("\n" + "="*80)
    print("MODEL:", model_id)

    df_hyp, df_eval, elapsed, w, c = transcribe_one_model(model_id, audio_files, df_ref)

    # 저장(메모리)
    df_hyp_by_model[model_id]  = df_hyp
    df_eval_by_model[model_id] = df_eval

    # 저장(파일) - 선택
    hyp_path = OUT_DIR / f"hyp_{model_id.replace('/','_')}.csv"
    eval_path = OUT_DIR / f"eval_{model_id.replace('/','_')}.csv"
    df_hyp.to_csv(hyp_path, index=False)
    df_eval.to_csv(eval_path, index=False)

    row = {
        "model_id": model_id,
        "n_files": len(audio_files),
        "elapsed_sec": round(elapsed, 3),
        "sec_per_file": round(elapsed / max(len(audio_files), 1), 4),
        "wer": round(w, 6),
        "cer": round(c, 6),
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    all_results.append(row)

    print("Saved hyp:", hyp_path)
    print("Saved eval:", eval_path)
    print("WER:", row["wer"], "| CER:", row["cer"], "| elapsed(s):", row["elapsed_sec"])



MODEL: openai/whisper-tiny
MODEL: openai/whisper-tiny | CUDA_VISIBLE_DEVICES: 1 | pipeline device: 0 | dtype: torch.float16


Device set to use cuda:0
`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.


ref rows: 456 | hyp rows: 456 | matched: 456
Saved hyp: /home/data/bootcamp/user2/gits/espnet-notebook-202601.new/bootcamp/outputs_asr/hyp_openai_whisper-tiny.csv
Saved eval: /home/data/bootcamp/user2/gits/espnet-notebook-202601.new/bootcamp/outputs_asr/eval_openai_whisper-tiny.csv
WER: 0.542887 | CER: 0.222445 | elapsed(s): 76.21

MODEL: openai/whisper-base
MODEL: openai/whisper-base | CUDA_VISIBLE_DEVICES: 1 | pipeline device: 0 | dtype: torch.float16


Device set to use cuda:0


ref rows: 456 | hyp rows: 456 | matched: 456
Saved hyp: /home/data/bootcamp/user2/gits/espnet-notebook-202601.new/bootcamp/outputs_asr/hyp_openai_whisper-base.csv
Saved eval: /home/data/bootcamp/user2/gits/espnet-notebook-202601.new/bootcamp/outputs_asr/eval_openai_whisper-base.csv
WER: 0.493658 | CER: 0.223784 | elapsed(s): 85.52

MODEL: openai/whisper-small
MODEL: openai/whisper-small | CUDA_VISIBLE_DEVICES: 1 | pipeline device: 0 | dtype: torch.float16


Device set to use cuda:0


ref rows: 456 | hyp rows: 456 | matched: 456
Saved hyp: /home/data/bootcamp/user2/gits/espnet-notebook-202601.new/bootcamp/outputs_asr/hyp_openai_whisper-small.csv
Saved eval: /home/data/bootcamp/user2/gits/espnet-notebook-202601.new/bootcamp/outputs_asr/eval_openai_whisper-small.csv
WER: 0.378133 | CER: 0.12254 | elapsed(s): 109.57


In [9]:
import pandas as pd
import os
from pathlib import Path

# 1) 모델별 df_eval들을 하나로 합치기 (long)
df_all = pd.concat(df_eval_by_model.values(), ignore_index=True)

# 안전장치: file은 basename 통일
df_all["file"] = df_all["file"].astype(str).apply(os.path.basename)
df_all["ref"]  = df_all["ref"].astype(str)
df_all["hyp"]  = df_all["hyp"].astype(str)

# 2) wide(파일 단위 비교)로 변환: (file, ref) 한 줄 + 모델별 hyp 컬럼
df_cmp = (
    df_all.pivot_table(
        index=["file", "ref"],
        columns="model_id",
        values="hyp",
        aggfunc="first"
    )
    .reset_index()
)

# 3) 컬럼명 정리: hyp__openai_whisper_base 형태
df_cmp.columns = [
    f"hyp__{c.replace('/', '_')}" if c not in ("file", "ref") else c
    for c in df_cmp.columns
]

# 3) df_cmp_view (또는 df)에 wav_path를 "실제 경로"로 채움
df_cmp_view = df_cmp.copy()
df_cmp_view["wav_path"] = df_cmp_view["file"].map(wav_map)

# 4) 검증
missing = df_cmp_view["wav_path"].isna().sum()
print("missing wav_path:", missing, "/", len(df_cmp_view))
print("example wav_path:", df_cmp_view["wav_path"].dropna().iloc[0])


# 4) 오디오 경로 붙이기
DATA_AUDIO_DIR_P = Path(DATA_AUDIO_DIR)  # DATA_AUDIO_DIR가 Path면 그대로 써도 됨
df_cmp["wav_path"] = df_cmp["file"].astype(str).apply(lambda fn: str(DATA_AUDIO_DIR_P / fn))

print("rows:", len(df_cmp))
print("cols:", df_cmp.columns.tolist())
df_cmp.head(10)


missing wav_path: 0 / 456
example wav_path: /home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_0019.wav
rows: 456
cols: ['file', 'ref', 'hyp__openai_whisper-base', 'hyp__openai_whisper-small', 'hyp__openai_whisper-tiny', 'wav_path']


,file,ref,hyp__openai_whisper-base,hyp__openai_whisper-small,hyp__openai_whisper-tiny,wav_path
0,104_003_0019.wav,지난해 삼 월 김 전 장관의 동료인 장동련 홍익대 교수가 민간 자문단장으로 위촉되면...,지난해 3월 김 전 장관의 동료인 장동량 홍익대 교수가 민간 자문단장으로 위쪽되면서...,지난해 3월 김전 장관의 동료인 장동량 홍익대 교수가 민간 자문단장으로 위촉되면서 ...,지난해 3월 김전 장관에 동료인 장동량 홍입대 교수가 민간 자문단장으로 위축되는서 ...,/home/data/bootcamp/user2/data_zeroth/test_dat...
1,104_003_0193.wav,그 바람에 나의 몸도 겹쳐 쓰러지며 한창 피어난 노란 동백꽃 속으로 폭 파묻혀 버렸다,그 바람에 나의 몸도 겹쳐스러지면 한참 피어난 노란 동백꽃속으로 폭파무처버렸다.,그 바람에 나의 몸도 겹쳐 쓰러지며 한참 피어난 노란 동백꽃속으로 폭 파묻혀 버렸다.,그 바람에 나의 몸도 겹쳐스러지면 한창 피형한 노란 동대꽃으로 폭팔을 쳐버렸다.,/home/data/bootcamp/user2/data_zeroth/test_dat...
2,104_003_0253.wav,현재 백화점과 영화관 등은 오픈해 영업하고 있고 테마파크 및 아파트 등의 공사는 이...,현재 백화점과 영화관 등은 오픈의 영화파고 있고 테마파크미 아파트 등의 공사는 20...,현재 백화점과 영화관 등은 오픈에 영업하고 있고 테마 파크 및 아파트 등의 공사는 ...,현재 백화정과 영어간 등은 오프에 영업하고 있고 테마파크미 아빛의 등의 공사는 20...,/home/data/bootcamp/user2/data_zeroth/test_dat...
3,104_003_0280.wav,또한 박 전 대통령 측은 앞으로 검찰 수사와 관련해 수사 내용에 대한 입장뿐 아니라...,또한 박 전 대통령 측은 앞으로 검찰 수사와 관련해 수사 내용에 대한 입장 뿐 아니...,또한 박 전 대통령 측은 앞으로 검찰 수사와 관련해 수사 내용에 대한 입장뿐 아니라...,또한 박정 대통령 측은 앞으로 검찰 수사와 관련해 수사 내용이 대한 입장 뿐 아니라...,/home/data/bootcamp/user2/data_zeroth/test_dat...
4,104_003_0294.wav,한 잔의 커피가 소비자에게 전해지기 위해선 여러 차례의 공정을 거쳐야 한다,한잔의 커피가 소비자에게 전해지기 위해서는 여러 차례의 공정을 거쳐야 한다.,한잔의 커피가 소비자에게 전해지기 위해서는 여러 차례의 공정을 거쳐야 한다.,환전에 커피가 속이 자에게 전해지기 위해서 여러 차례의 공정을 거쳐야 한다.,/home/data/bootcamp/user2/data_zeroth/test_dat...
5,104_003_0352.wav,다른 지역에서는 그곳이 아무리 넓다 하더라도 푸야 라이몬디를 발견하기가 어려울 것입니다,다른 지역에서는 그곳이 아무리 넓다 하더라도 푸야 라인 문제를 발견하기가 어려울 것...,다른 지역에서는 그곳이 아무리 넓다 하더라도 푸야 라이문디를 발견하기가 어려울 것입니다.,다른 지역에서는 규모시에 아무리 넓다 하더라도 후야 라인 문기를 발견하기가 어려울 ...,/home/data/bootcamp/user2/data_zeroth/test_dat...
6,104_003_0360.wav,물론 십 대 자녀가 잘못된 결론을 내리거나 심지어 당신과 비슷한 실수를 저질렀을 때...,물론 10대 자녀가 잘못된 결론을 내리거나 심지어 당신과 비슷한 실수를 저질렀을 때...,물론 10대 자녀가 잘못된 결론을 내리거나 심지어 당신과 비슷한 실수를 저질렀을 때...,물론 식대 전혀가 잘못된 결론을 내리거나 심지어 당신과 비슷한 실수를 전질했을 때 ...,/home/data/bootcamp/user2/data_zeroth/test_dat...
7,104_003_0374.wav,만약 박근혜 대통령이 정말로 손을 내밀면 안철수 대표와 국민의당은 어떤 선택을 할까요,만약 박근혜 대통령이 정말로 손을 내밀면 안철수 대표와 국민의당은 어떤 선택을 할까요?,만약 박근혜 대통령이 정말로 손을 내밀면 안철수 대표와 국민 의당은 어떤 선택을 할까요?,만약 바꾼에 대통령이 정말로 손을 내리면 안철수 대표와 국민을 닿는 어떤 선택을 할까요?,/home/data/bootcamp/user2/data_zeroth/test_dat...
8,104_003_0437.wav,대부분 전기는 기업이 사용 가정용 전기에 누진제 적용하는 국가 거의 없어 많이 쓴다...,"대부분 전기는 기업이 사용, 가정용 전기의 요진제 적용하는 국가 거의 없어. 많이 ...","대부분 전기는 기업이 사용, 가정용 전기에 누진제 적용하는 국가 거의 없어 많이 쓴...","대부분 정기는 기업이 사용, 가정용 정기에 요즘 제 정용하는 국가 거의 없어, 많이...",/home/data/bootcamp/user2/data_zeroth/test_dat...
9,104_003_0451.wav,더 선은 이름이 밝혀지지 않은 이 노인이 이 년 전 피부암을 진단받은 뒤 자신의 양...,더서는 이름이 밝혀지지 않은 이 노인이 2년 전 피부함을 짓는 받은 게 자신의 양신...,더 선은 이름이 밝혀지지 않은 이 노인이 2년 전 피부함을 짓난받은 뒤 자신의 양신...,더 선은 이름이 밝혀지지 않은 이 노인이 인연장 피부함을 진난 받은 게 자신의 양신...,/home/data/bootcamp/user2/data_zeroth/test_dat...


In [10]:
import pandas as pd
from jiwer import wer, cer

# df_cmp: [file, ref, wav_path, hyp__...] 형태여야 함
hyp_cols = [c for c in df_cmp.columns if c.startswith("hyp__")]
assert len(hyp_cols) > 0, "hyp__ 컬럼이 없습니다. df_cmp pivot 셀부터 확인하세요."

def safe_wer(ref, hyp):
    ref = "" if pd.isna(ref) else str(ref)
    hyp = "" if pd.isna(hyp) else str(hyp)
    return wer([ref], [hyp])

def safe_cer(ref, hyp):
    ref = "" if pd.isna(ref) else str(ref)
    hyp = "" if pd.isna(hyp) else str(hyp)
    return cer([ref], [hyp])

# (주의) apply row-wise는 느릴 수 있음. 지금 50개면 충분히 OK.
for hyp_c in hyp_cols:
    tag = hyp_c.replace("hyp__", "")          # openai_whisper_base 같은 형태
    df_cmp[f"wer__{tag}"] = df_cmp.apply(lambda r: safe_wer(r["ref"], r[hyp_c]), axis=1)
    df_cmp[f"cer__{tag}"] = df_cmp.apply(lambda r: safe_cer(r["ref"], r[hyp_c]), axis=1)

# 평균 CER 기준으로 모델 정렬
cer_cols = [c for c in df_cmp.columns if c.startswith("cer__")]
model_order = sorted(cer_cols, key=lambda cc: df_cmp[cc].mean())

# 보기용 컬럼 재배열: file/ref/wav_path + (모델별 cer/wer/hyp 묶음)
base_cols = ["file", "ref", "wav_path"]
cols = base_cols.copy()

for cer_c in model_order:
    tag = cer_c.replace("cer__", "")
    wer_c = f"wer__{tag}"
    hyp_c = f"hyp__{tag}"
    cols += [cer_c, wer_c, hyp_c]

df_cmp_view = df_cmp[cols].copy()

# 요약(모델별 평균 WER/CER)
summary = []
for cer_c in model_order:
    tag = cer_c.replace("cer__", "")
    summary.append({
        "model": tag.replace("_", "/"),
        "avg_cer": float(df_cmp[f"cer__{tag}"].mean()),
        "avg_wer": float(df_cmp[f"wer__{tag}"].mean()),
    })
df_summary = pd.DataFrame(summary).sort_values("avg_cer")

print(df_summary)
df_cmp_view.head(10)


                  model   avg_cer   avg_wer
0  openai/whisper-small  0.120541  0.373268
1   openai/whisper-base  0.218461  0.491543
2   openai/whisper-tiny  0.220773  0.540921


,file,ref,wav_path,cer__openai_whisper-small,wer__openai_whisper-small,hyp__openai_whisper-small,cer__openai_whisper-base,wer__openai_whisper-base,hyp__openai_whisper-base,cer__openai_whisper-tiny,wer__openai_whisper-tiny,hyp__openai_whisper-tiny
0,104_003_0019.wav,지난해 삼 월 김 전 장관의 동료인 장동련 홍익대 교수가 민간 자문단장으로 위촉되면...,/home/data/bootcamp/user2/data_zeroth/test_dat...,0.073529,0.333333,지난해 3월 김전 장관의 동료인 장동량 홍익대 교수가 민간 자문단장으로 위촉되면서 ...,0.073529,0.277778,지난해 3월 김 전 장관의 동료인 장동량 홍익대 교수가 민간 자문단장으로 위쪽되면서...,0.161765,0.611111,지난해 3월 김전 장관에 동료인 장동량 홍입대 교수가 민간 자문단장으로 위축되는서 ...
1,104_003_0193.wav,그 바람에 나의 몸도 겹쳐 쓰러지며 한창 피어난 노란 동백꽃 속으로 폭 파묻혀 버렸다,/home/data/bootcamp/user2/data_zeroth/test_dat...,0.063830,0.285714,그 바람에 나의 몸도 겹쳐 쓰러지며 한참 피어난 노란 동백꽃속으로 폭 파묻혀 버렸다.,0.212766,0.571429,그 바람에 나의 몸도 겹쳐스러지면 한참 피어난 노란 동백꽃속으로 폭파무처버렸다.,0.297872,0.571429,그 바람에 나의 몸도 겹쳐스러지면 한창 피형한 노란 동대꽃으로 폭팔을 쳐버렸다.
2,104_003_0253.wav,현재 백화점과 영화관 등은 오픈해 영업하고 있고 테마파크 및 아파트 등의 공사는 이...,/home/data/bootcamp/user2/data_zeroth/test_dat...,0.176471,0.450000,현재 백화점과 영화관 등은 오픈에 영업하고 있고 테마 파크 및 아파트 등의 공사는 ...,0.220588,0.500000,현재 백화점과 영화관 등은 오픈의 영화파고 있고 테마파크미 아파트 등의 공사는 20...,0.279412,0.600000,현재 백화정과 영어간 등은 오프에 영업하고 있고 테마파크미 아빛의 등의 공사는 20...
3,104_003_0280.wav,또한 박 전 대통령 측은 앞으로 검찰 수사와 관련해 수사 내용에 대한 입장뿐 아니라...,/home/data/bootcamp/user2/data_zeroth/test_dat...,0.012987,0.047619,또한 박 전 대통령 측은 앞으로 검찰 수사와 관련해 수사 내용에 대한 입장뿐 아니라...,0.064935,0.190476,또한 박 전 대통령 측은 앞으로 검찰 수사와 관련해 수사 내용에 대한 입장 뿐 아니...,0.155844,0.333333,또한 박정 대통령 측은 앞으로 검찰 수사와 관련해 수사 내용이 대한 입장 뿐 아니라...
4,104_003_0294.wav,한 잔의 커피가 소비자에게 전해지기 위해선 여러 차례의 공정을 거쳐야 한다,/home/data/bootcamp/user2/data_zeroth/test_dat...,0.097561,0.363636,한잔의 커피가 소비자에게 전해지기 위해서는 여러 차례의 공정을 거쳐야 한다.,0.097561,0.363636,한잔의 커피가 소비자에게 전해지기 위해서는 여러 차례의 공정을 거쳐야 한다.,0.219512,0.545455,환전에 커피가 속이 자에게 전해지기 위해서 여러 차례의 공정을 거쳐야 한다.
5,104_003_0352.wav,다른 지역에서는 그곳이 아무리 넓다 하더라도 푸야 라이몬디를 발견하기가 어려울 것입니다,/home/data/bootcamp/user2/data_zeroth/test_dat...,0.041667,0.181818,다른 지역에서는 그곳이 아무리 넓다 하더라도 푸야 라이문디를 발견하기가 어려울 것입니다.,0.104167,0.272727,다른 지역에서는 그곳이 아무리 넓다 하더라도 푸야 라인 문제를 발견하기가 어려울 것...,0.208333,0.454545,다른 지역에서는 규모시에 아무리 넓다 하더라도 후야 라인 문기를 발견하기가 어려울 ...
6,104_003_0360.wav,물론 십 대 자녀가 잘못된 결론을 내리거나 심지어 당신과 비슷한 실수를 저질렀을 때...,/home/data/bootcamp/user2/data_zeroth/test_dat...,0.040000,0.150000,물론 10대 자녀가 잘못된 결론을 내리거나 심지어 당신과 비슷한 실수를 저질렀을 때...,0.040000,0.150000,물론 10대 자녀가 잘못된 결론을 내리거나 심지어 당신과 비슷한 실수를 저질렀을 때...,0.106667,0.300000,물론 식대 전혀가 잘못된 결론을 내리거나 심지어 당신과 비슷한 실수를 전질했을 때 ...
7,104_003_0374.wav,만약 박근혜 대통령이 정말로 손을 내밀면 안철수 대표와 국민의당은 어떤 선택을 할까요,/home/data/bootcamp/user2/data_zeroth/test_dat...,0.042553,0.250000,만약 박근혜 대통령이 정말로 손을 내밀면 안철수 대표와 국민 의당은 어떤 선택을 할까요?,0.021277,0.083333,만약 박근혜 대통령이 정말로 손을 내밀면 안철수 대표와 국민의당은 어떤 선택을 할까요?,0.191489,0.416667,만약 바꾼에 대통령이 정말로 손을 내리면 안철수 대표와 국민을 닿는 어떤 선택을 할까요?
8,104_003_0437.wav,대부분 전기는 기업이 사용 가정용 전기에 누진제 적용하는 국가 거의 없어 많이 쓴다...,/home/data/bootcamp/user2/data_zeroth/test_dat...,0.015385,0.052632,"대부분 전기는 기업이 사용, 가정용 전기에 누진제 적용하는 국가 거의 없어 많이 쓴...",0.076923,0.263158,"대부분 전기는 기업이 사용, 가정용 전기의 요진제 적용하는 국가 거의 없어. 많이 ...",0.169231,0.473684,"대부분 정기는 기업이 사용, 가정용 정기에 요즘 제 정용하는 국가 거의 없어, 많이..."
9,104_003_0451.wav,더 선은 이름이 밝혀지지 않은 이 노인이 이 년 전 피부암을 진단받은 뒤 자신의 양...,/home/data/bootcamp/user2/data_zeroth/test_dat...,0.118644,0.352941,더 선은 이름이 밝혀지지 않은 이 노인이 2년 전 피부함을 짓난받은 뒤 자신의 양신...,0.237288,0.647059,더서는 이름이 밝혀지지 않은 이 노인이 2년 전 피부함을 짓는 받은 게 자신의 양신...,0.254237,0.588235,더 선은 이름이 밝혀지지 않은 이 노인이 인연장 피부함을 진난 받은 게 자신의 양신...


In [11]:
from pathlib import Path
import glob, os
from IPython.display import Audio, display

# 파일 스캔은 1회만
audio_files_all = [Path(p) for p in glob.glob(os.path.join(DATA_AUDIO_DIR, "**", "*.wav"), recursive=True)]
wav_map = {p.name: str(p) for p in audio_files_all}


In [12]:
from IPython.display import Audio, display

def resolve_wav_path(p):
    # p가 이미 full path이면 그대로
    if p and os.path.exists(p):
        return p
    # basename이면 map에서 찾기
    base = os.path.basename(str(p))
    return wav_map.get(base)

# 가장 성능 좋은 모델(평균 CER 최소) tag
best_tag = df_summary.iloc[0]["model"].replace("/", "_")
cer_col = f"cer__{best_tag}"

# CER 큰 순으로 5개
df_pick = df_cmp_view.sort_values(cer_col, ascending=False).head(5)

for _, r in df_pick.iterrows():
    print("=" * 90)
    print("FILE:", r["file"])
    print("PATH:", r["wav_path"])
    print("REF :", r["ref"])
    print("-" * 90)

    # 모델별 출력(현재 df_cmp_view 컬럼 순서가 이미 성능 순 정렬됨)
    for col in [c for c in df_cmp_view.columns if c.startswith("hyp__")]:
        tag = col.replace("hyp__", "")
        print(f"[{tag.replace('_','/')}] CER={r.get('cer__'+tag):.4f}  WER={r.get('wer__'+tag):.4f}")
        print(r[col])
        print()
    wav_path = resolve_wav_path(r["wav_path"])
    display(Audio(filename=wav_path))


FILE: 126_003_0376.wav
PATH: /home/data/bootcamp/user2/data_zeroth/test_data_01/126_003_0376.wav
REF : 반면 하종강 성공회대 노동아카데미 주임교수는 인간에 대한 이해 부족이 낳은 현상이라며 실직하는 노동자의 비인간적 고통을 생각해야 한다고 강조했다
------------------------------------------------------------------------------------------
[openai/whisper-small] CER=0.7125  WER=0.9444
반면 하중간 성공에 대해 노동 아카데미 주인 교수는 인간에 대해

[openai/whisper-base] CER=0.7375  WER=0.9444
반면 사중간 성공에 대해 노동아카데미 주인교수는 인간회대에

[openai/whisper-tiny] CER=0.7250  WER=0.9444
반면 다존감 성공에 대해 노동 아카데미 주인 교수는 인간에 대해



FILE: 126_003_0681.wav
PATH: /home/data/bootcamp/user2/data_zeroth/test_data_01/126_003_0681.wav
REF : 삼 분기에만 매출 일 조 이천 구백 십 구 억원 영업이익 육백 십 팔 억원을 올렸다
------------------------------------------------------------------------------------------
[openai/whisper-small] CER=0.5652  WER=0.8750
3분기 매출 1초 2919원 영업이 618억원을 올렸다

[openai/whisper-base] CER=0.5652  WER=0.9375
3분기 매출 1조 2919원 영업이 618억 원을 올렸다.

[openai/whisper-tiny] CER=0.5435  WER=0.8750
3분기 매출 1조 2천 919원 영업이 680원에 올렸다



FILE: 149_003_2621.wav
PATH: /home/data/bootcamp/user2/data_zeroth/test_data_01/149_003_2621.wav
REF : 경남에서도 열 여덟 개 시 군 가운데 열 한 개 시 군만 월 삼 만 십 만원의 수당을 준다
------------------------------------------------------------------------------------------
[openai/whisper-small] CER=0.5000  WER=0.7895
경남에서도 18개씩은 가운데 11개씩은 만 5월 30만원에 수당을 준다.

[openai/whisper-base] CER=0.4400  WER=0.8421
경남에서도 18개 시금 가운데 11개 시금만 오해 30만 원을 수당을 준다.

[openai/whisper-tiny] CER=0.5800  WER=0.8947
정당에서도 16개씩은 가운데 11개씩은 한개씩은 10만 원 30만 원에 수당을 준다.



FILE: 137_003_1848.wav
PATH: /home/data/bootcamp/user2/data_zeroth/test_data_01/137_003_1848.wav
REF : 회수대상은 이천 십 육 년 시 월 일 일 에서 이천 십 육 년 십 일 월 십 구 일 내로 유통기한이 표시된 제품들이다
------------------------------------------------------------------------------------------
[openai/whisper-small] CER=0.4769  WER=0.9167
해수 대상은 2016년 10월 1일인에서 2016년 11월 19일 내로 유통기관이 표시된 제품들이다.

[openai/whisper-base] CER=0.5077  WER=0.9583
해소 대상은 2016년 시월 1일에서 2016년 11월 10고인 앨로 유통기안이 표시된 제품들이다.

[openai/whisper-tiny] CER=0.4923  WER=0.9583
회수 대상은 2016년 10월 1일에서 2016년 11월 19일으로 유통기환이 표시된 제품들이다.



FILE: 147_003_2999.wav
PATH: /home/data/bootcamp/user2/data_zeroth/test_data_01/147_003_2999.wav
REF : 국립 박물관의 연구원 테레자 본토르치히는 자신의 저서 믿음 때문에 투옥되다 아우슈비츠 강제 수용소의 여호와의 증인 에서 이렇게 썼습니다
------------------------------------------------------------------------------------------
[openai/whisper-small] CER=0.4667  WER=0.8333
국립박물관이 연구한 테레자 본토르 지인은 자신에 져서 믿음 때문에 투옥되다 아우스비츠 강재수요.

[openai/whisper-base] CER=0.4533  WER=0.7222
국립 박물 많이 연고한 테레자 본토르 지인은 자신의 저서 믿음 때문에 투억되다 아웃수비츠 강제 수용

[openai/whisper-tiny] CER=0.4667  WER=0.7222
국립 방물관이 연구한 테레자번토를 지인한 자신의 저서 믿음 때문에 투업되다 아우스비츠 강제수용

